# Unidade 3 — Aula 3
## Criação de novos atributos
Ciência de Dados · Bacharelado em Engenharia de Software · UNIFSA


**Situação:** o dono de um restaurante quer saber quais clientes dão as melhores gorjetas e como organizar o atendimento. As colunas originais não respondem diretamente a essas perguntas. Nesta atividade, você vai criar novos atributos a partir das colunas existentes.

**Base de dados:** *Tips* (BRYANT, P. G.; SMITH, M. A., 1995). 244 contas registradas por um garçom em um restaurante dos Estados Unidos, com valores em dólares. A base é carregada de uma cópia pública:
https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv


## 1. Carga e preparação da base

In [52]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
df = pd.read_csv(URL)
df.columns = ["conta", "gorjeta", "sexo", "fumante", "dia", "turno", "pessoas"]
df = df.drop_duplicates().reset_index(drop=True)
df.head()

,conta,gorjeta,sexo,fumante,dia,turno,pessoas
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


| Coluna | Descrição |
|---|---|
| conta | valor da conta, em dólares, sem a gorjeta |
| gorjeta | valor da gorjeta, em dólares |
| sexo | sexo de quem pagou a conta |
| fumante | se havia fumantes na mesa (Yes, No) |
| dia | dia da semana (Thur, Fri, Sat, Sun) |
| turno | almoço (Lunch) ou jantar (Dinner) |
| pessoas | número de pessoas na mesa |

## 2. Razão: dividir uma coluna por outra
**Exemplo:** percentual da gorjeta em relação à conta, e sua média por tamanho de grupo.

In [53]:
df["percentual_gorjeta"] = df["gorjeta"] / df["conta"] * 100
df.groupby("pessoas")[["gorjeta", "percentual_gorjeta"]].mean().round(2)

,gorjeta,percentual_gorjeta
pessoas,,
1,1.44,21.73
2,2.59,16.58
3,3.39,15.22
4,4.14,14.59
5,4.03,14.15
6,5.22,15.62


**Sua vez 2.1:** crie o atributo `conta_por_pessoa` (valor da conta dividido pelo número de pessoas) e calcule sua média por turno.

In [54]:
df["conta_por_pessoa"] = df["conta"] / df["pessoas"]
df.groupby("pessoas")[["conta", "conta_por_pessoa"]].mean().round(2)

,conta,conta_por_pessoa
pessoas,,
1,7.24,7.24
2,16.47,8.24
3,23.28,7.76
4,28.61,7.15
5,30.07,6.01
6,34.83,5.80


**Pergunta 2.2:** em qual turno cada cliente gasta mais, individualmente? Que diferença existe entre olhar `conta` e olhar `conta_por_pessoa`?

In [55]:
turno_gastos = df.groupby("turno")[["conta", "conta_por_pessoa"]].mean().round(2)

print("Os gastos cada turno: ")
print(turno_gastos)

o_mais_mais = turno_gastos["conta_por_pessoa"].idxmax()
print(f"\nO turno que tem mais gastos: {o_mais_mais}")

Os gastos cada turno: 
        conta  conta_por_pessoa
turno                          
Dinner  20.80              8.11
Lunch   17.23              7.33

O turno que tem mais gastos: Dinner


**Resposta:**



Cada cliente gasta mais individualmente no jantar (Dinner), com média de 8,11 por pessoa face a 7,33 no almoço (Lunch). Olhar apenas conta mostra o valor total da mesa (20,80 no jantar vs. 17,23 no almoço), que sofre influência do número de pessoas; já conta_por_pessoa divide o valor pelo tamanho do grupo e evidencia o consumo médio individual.

## 3. Soma: combinar colunas na mesma unidade
**Exemplo:** total efetivamente pago pelo cliente.

In [56]:
df["total_pago"] = df["conta"] + df["gorjeta"]
df[["conta", "gorjeta", "total_pago"]].head()

,conta,gorjeta,total_pago
0,16.99,1.01,18.00
1,10.34,1.66,12.00
2,21.01,3.50,24.51
3,23.68,3.31,26.99
4,24.59,3.61,28.20


**Sua vez 3.1:** crie o atributo `total_por_pessoa` (total pago dividido pelo número de pessoas) e mostre as cinco mesas com maior valor.

In [57]:
df["total_pago"] = df["conta"] + df["gorjeta"]

df["total_por_pessoa"] = (df["total_pago"] / df["pessoas"])

df.nlargest(5, "total_por_pessoa")[["conta", "gorjeta", "pessoas", "total_pago", "total_por_pessoa"]].round(2)

,conta,gorjeta,pessoas,total_pago,total_por_pessoa
184,40.55,3.00,2,43.55,21.78
170,50.81,10.00,3,60.81,20.27
179,34.63,3.55,2,38.18,19.09
83,32.68,5.00,2,37.68,18.84
175,32.90,3.11,2,36.01,18.00


## 4. Indicador: transformar uma condição em 0 ou 1
**Exemplo:** a mesa foi atendida no fim de semana? Comparação do percentual médio da gorjeta.

In [58]:
df["fim_de_semana"] = df["dia"].isin(["Sat", "Sun"]).astype(int)
df.groupby("fim_de_semana")["percentual_gorjeta"].agg(["count", "mean"]).round(2)

,count,mean
fim_de_semana,,
0,80,16.34
1,163,15.96


**Sua vez 4.1:** crie o indicador `grupo_grande`, igual a 1 para mesas com 4 pessoas ou mais e 0 nos demais casos. Compare o percentual médio da gorjeta entre os dois grupos.

In [59]:
df["grupo_grande"] = (df["pessoas"] >= 4).astype(int)
df.groupby("grupo_grande")["percentual_gorjeta"].agg(["count", "mean"]).round(2)

,count,mean
grupo_grande,,
0,197,16.42
1,46,14.64


**Pergunta 4.2:** o fim de semana e o tamanho do grupo influenciam o percentual da gorjeta da mesma forma? Justifique com os números.

In [60]:
resultado_fim_semana = df.groupby("fim_de_semana")["percentual_gorjeta"].mean()
resultado_grupo = df.groupby("grupo_grande")["percentual_gorjeta"].mean()

print(resultado_fim_semana.round(2))
print(resultado_grupo.round(2))


fim_de_semana
0    16.34
1    15.96
Name: percentual_gorjeta, dtype: float64
grupo_grande
0    16.42
1    14.64
Name: percentual_gorjeta, dtype: float64


**Resposta:**



Não influenciam da mesma forma. O fim de semana tem impacto reduzido no percentual médio da gorjeta (15,96% no fim de semana vs. 16,34% nos outros dias, diferença de -0,38 pontos percentuais). Já o tamanho do grupo tem influência mais expressiva: mesas com 4 ou mais pessoas apresentam percentual médio menor (14,64%) do que mesas menores (16,42%, diferença de -1,78 pontos percentuais).

## 5. Faixa: transformar um número em categorias ordenadas
**Exemplo:** faixas de valor da conta com cortes fixos (até 15, de 15 a 25, acima de 25 dólares).

In [61]:
df["faixa_conta"] = pd.cut(df["conta"], bins=[0, 15, 25, np.inf], labels=["baixa", "média", "alta"])
df.groupby("faixa_conta", observed=True)["percentual_gorjeta"].agg(["count", "mean"]).round(2)

,count,mean
faixa_conta,,
baixa,79,18.39
média,109,15.88
alta,55,13.17


**Sua vez 5.1:** crie o atributo `faixa_conta_igual` usando `pd.qcut`, que divide as mesas em três grupos com a mesma quantidade de mesas. Mostre o menor e o maior valor de conta de cada faixa e o percentual médio da gorjeta.

In [62]:
df["faixa_conta_igual"] = pd.qcut(df["conta"], q=3, labels=["baixa", "média", "alta"])

resultado_faixas = df.groupby("faixa_conta_igual", observed=True).agg(
    menor_conta=("conta", "min"),
    maior_conta=("conta", "max"),
    percentual_medio=("percentual_gorjeta", "mean"),
    quantidade=("conta", "count")
).round(2)

resultado_faixas

,menor_conta,maior_conta,percentual_medio,quantidade
faixa_conta_igual,,,,
baixa,3.07,15.04,18.27,81
média,15.06,21.01,16.17,81
alta,21.16,50.81,13.81,81


**Pergunta 5.2:** qual das duas formas de criar faixas você apresentaria ao dono do restaurante? Por quê?

Escolho as faixas criadas com pd.cut(), pois elas utilizam valores fixos de contas, facilitando a interpretação e planejamento do restaurante.

Porque o proprietário consegue identificar facilmente clientes com contas baixas (até US$ 15), médias (de US$ 15 a US$ 25) e altas (acima de US$ 25).

O pd.qcut() também é útil, principalmente quando queremos dividir os clientes em grupos com quantidades semelhantes de registros. Porém, seus limites podem ser menos intuitivos e variar conforme os dados.

**Resposta:**



## 6. Agregação: comparar cada mesa com o seu grupo
**Exemplo:** percentual médio da gorjeta do dia de cada mesa e indicador de mesa acima da média do seu dia.

In [63]:
df["media_do_dia"] = df.groupby("dia")["percentual_gorjeta"].transform("mean")
df["acima_media_dia"] = (df["percentual_gorjeta"] > df["media_do_dia"]).astype(int)
df[["dia", "percentual_gorjeta", "media_do_dia", "acima_media_dia"]].head().round(2)

,dia,percentual_gorjeta,media_do_dia,acima_media_dia
0,Sun,5.94,16.69,0
1,Sun,16.05,16.69,0
2,Sun,16.66,16.69,0
3,Sun,13.98,16.69,0
4,Sun,14.68,16.69,0


**Sua vez 6.1:** crie os atributos `media_do_turno` e `acima_media_turno`, seguindo o exemplo. Em seguida, conte quantas mesas de cada turno ficaram acima da média do próprio turno.

In [64]:
df["media_do_turno"] = df.groupby("turno")["percentual_gorjeta"].transform("mean")
df["acima_media_turno"] = (df["percentual_gorjeta"] > df["media_do_turno"]).astype(int)
df[["turno", "percentual_gorjeta","media_do_turno", "acima_media_turno"]].head().round(2)

resultado_turno = df.groupby("turno")["acima_media_turno"].agg(["sum", "count"])
resultado_turno.columns = ["mesas_acima_media","total_mesas"]

resultado_turno

,mesas_acima_media,total_mesas
turno,,
Dinner,80,176
Lunch,30,67


## 7. Os novos atributos e as aulas anteriores
**Exemplo:** padronização do percentual da gorjeta para identificar mesas fora do padrão.

In [65]:
df["percentual_z"] = (df["percentual_gorjeta"] - df["percentual_gorjeta"].mean()) / df["percentual_gorjeta"].std()
df.nlargest(3, "percentual_z")[["conta", "gorjeta", "pessoas", "percentual_gorjeta", "percentual_z"]].round(2)

,conta,gorjeta,pessoas,percentual_gorjeta,percentual_z
172,7.25,5.15,2,71.03,8.98
178,9.60,4.00,2,41.67,4.18
67,3.07,1.00,1,32.57,2.69


**Sua vez 7.1:** calcule a correlação de `conta`, `pessoas` e `conta_por_pessoa` com a coluna `gorjeta` e, separadamente, com o atributo `percentual_gorjeta`.

In [66]:
atributos = ["conta", "pessoas", "conta_por_pessoa"]

correlacao_gorjeta = df[atributos + ["gorjeta"]].corr()
correlacao_percentual = df[atributos + ["percentual_gorjeta"]].corr()

print("Correlação com gorjeta: ")
display(correlacao_gorjeta["gorjeta"].round(3))

print("Correlação com percentual de gorjeta: ")
display(correlacao_percentual["percentual_gorjeta"].round(3))

Correlação com gorjeta: 


conta               0.675
pessoas             0.488
conta_por_pessoa    0.347
gorjeta             1.000
Name: gorjeta, dtype: float64

Correlação com percentual de gorjeta: 


conta                -0.339
pessoas              -0.143
conta_por_pessoa     -0.315
percentual_gorjeta    1.000
Name: percentual_gorjeta, dtype: float64

**Pergunta 7.2:** a conta se relaciona com a gorjeta e com o percentual da gorjeta da mesma forma? O que isso significa para o dono do restaurante?

Não se relacionam da mesma forma. O valor da conta apresenta correlação positiva com o valor absoluto da gorjeta (0,675), mas correlação negativa com o percentual da gorjeta (-0,339): contas maiores costumam gerar mais dinheiro em gorjeta, porém um percentual menor sobre o valor total. Para o dono do restaurante, isso significa analisar os dois indicadores em conjunto para não confundir maior volume de gorjeta com maior generosidade percentual.

**Resposta:**



## 8. Atributo próprio
**Sua vez 8.1:** crie um atributo novo que não tenha sido feito nesta aula. Escreva, na célula de resposta abaixo do código, a pergunta do dono do restaurante que ele responde e qual dos tipos vistos ele usa (razão, soma, indicador, faixa ou agregação).

**Resposta:**



In [67]:
df["gorjeta_por_pessoa"] = (df["gorjeta"] / df["pessoas"])
df[["conta", "gorjeta", "pessoas", "gorjeta_por_pessoa"]].head().round(2)

df.nlargest(5,"gorjeta_por_pessoa")[["conta", "gorjeta", "pessoas","gorjeta_por_pessoa"]].round(2)

,conta,gorjeta,pessoas,gorjeta_por_pessoa
170,50.81,10.00,3,3.33
88,24.71,5.85,2,2.92
181,23.33,5.65,2,2.82
172,7.25,5.15,2,2.58
46,22.23,5.00,2,2.50


Pergunta: "Quanto cada pessao da mesa contribui, em média, em valor de gorjeta?"

O tipo de atributo utilizado é **razão** (gorjeta / pessoas), pois divide o valor da gorjeta pelo número de pessoas. Esse indicador ajuda o restaurante a compreender quanto cada integrante contribui para a gorjeta total.

**Pergunta 8.2:** com base nos atributos criados, escreva uma recomendação de até três linhas para o dono do restaurante.

**Resposta:**



Recomendo acompanhar tanto o valor total das gorjetas quanto o percentual e a média por pessoa, pois esses indicadores permitem identificar diferentes padrões de consumo. Também é importante comparar os resultados por turno e tamanho dos grupos para organizar melhor o atendimento. O restaurante deve utilizar essas informações para melhorar a distribuição dos funcionários e a qualidade do serviço.